# Assessment 1 Phase 2: Historical Airline Data Analysis with Apache Spark

## 1. Environment Setup

This section initialises the Apache Spark environment used for the analysis and confirms the Spark version available in the Docker-based Jupyter environment.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Assessment1_Phase2") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.5


## 2. Dataset Loading and Initial Inspection

The approved U.S. airline on-time performance dataset is loaded into a Spark DataFrame. Initial checks are performed to confirm the number of records, number of columns, schema, and sample values before data cleaning and analysis.

In [2]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType
)

airline_schema = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("QUARTER", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY_OF_MONTH", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("FL_DATE", StringType(), True),
    StructField("OP_UNIQUE_CARRIER", StringType(), True),
    StructField("TAIL_NUM", StringType(), True),
    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
    StructField("ORIGIN_AIRPORT_ID", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("ORIGIN_CITY_NAME", StringType(), True),
    StructField("ORIGIN_STATE_ABR", StringType(), True),
    StructField("DEST_AIRPORT_ID", IntegerType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY_NAME", StringType(), True),
    StructField("DEST_STATE_ABR", StringType(), True),
    StructField("CRS_DEP_TIME", IntegerType(), True),
    StructField("DEP_TIME", IntegerType(), True),
    StructField("DEP_DELAY", DoubleType(), True),
    StructField("DEP_DELAY_NEW", DoubleType(), True),
    StructField("DEP_DEL15", DoubleType(), True),
    StructField("TAXI_OUT", DoubleType(), True),
    StructField("TAXI_IN", DoubleType(), True),
    StructField("CRS_ARR_TIME", IntegerType(), True),
    StructField("ARR_TIME", IntegerType(), True),
    StructField("ARR_DELAY", DoubleType(), True),
    StructField("ARR_DELAY_NEW", DoubleType(), True),
    StructField("ARR_DEL15", DoubleType(), True),
    StructField("CANCELLED", DoubleType(), True),
    StructField("DIVERTED", DoubleType(), True),
    StructField("CRS_ELAPSED_TIME", DoubleType(), True),
    StructField("ACTUAL_ELAPSED_TIME", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("FLIGHTS", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True)
])

file_path = "../T_ONTIME_REPORTING.csv"

df = (
    spark.read
    .option("header", True)
    .schema(airline_schema)
    .csv(file_path)
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 539747
Number of columns: 36


### 2.1 Initial Spark Partitioning

The initial number of Spark partitions is examined to establish a baseline for later performance analysis. Spark processes partitions in parallel, so the number and distribution of partitions can influence execution efficiency.

In [3]:
print("Initial number of partitions:", df.rdd.getNumPartitions())

Initial number of partitions: 8


In [4]:
df.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)


In [5]:
df.show(5, truncate=False)

+----+-------+-----+------------+-----------+--------------------+-----------------+--------+-----------------+-----------------+------+-----------------+----------------+---------------+----+-----------------+--------------+------------+--------+---------+-------------+---------+--------+-------+------------+--------+---------+-------------+---------+---------+--------+----------------+-------------------+--------+-------+--------+
|YEAR|QUARTER|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|FL_DATE             |OP_UNIQUE_CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN_AIRPORT_ID|ORIGIN|ORIGIN_CITY_NAME |ORIGIN_STATE_ABR|DEST_AIRPORT_ID|DEST|DEST_CITY_NAME   |DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|DEP_DELAY_NEW|DEP_DEL15|TAXI_OUT|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|
+----+-------+-----+------------+-----------+--------------------+-----------------+--------+-----------------+---------------

## 3. Data Quality Assessment and Cleaning

The dataset is examined for missing values, incorrect data type, and records that may require cleaning before analytical queries are performed.

In [6]:
for c in df.columns:
    null_count = df.filter(F.col(c).isNull()).count()
    print(c, null_count)

YEAR 0
QUARTER 0
MONTH 0
DAY_OF_MONTH 0
DAY_OF_WEEK 0
FL_DATE 0
OP_UNIQUE_CARRIER 0
TAIL_NUM 2530
OP_CARRIER_FL_NUM 0
ORIGIN_AIRPORT_ID 0
ORIGIN 0
ORIGIN_CITY_NAME 0
ORIGIN_STATE_ABR 0
DEST_AIRPORT_ID 0
DEST 0
DEST_CITY_NAME 0
DEST_STATE_ABR 0
CRS_DEP_TIME 0
DEP_TIME 15886
DEP_DELAY 15923
DEP_DELAY_NEW 15923
DEP_DEL15 15923
TAXI_OUT 16227
TAXI_IN 16580
CRS_ARR_TIME 0
ARR_TIME 16580
ARR_DELAY 17478
ARR_DELAY_NEW 17478
ARR_DEL15 17478
CANCELLED 0
DIVERTED 0
CRS_ELAPSED_TIME 0
ACTUAL_ELAPSED_TIME 17478
AIR_TIME 17478
FLIGHTS 0
DISTANCE 0


In [7]:
df.groupBy("DIVERTED").count().show()

+--------+------+
|DIVERTED| count|
+--------+------+
|     0.0|538581|
|     1.0|  1166|
+--------+------+



In [8]:
df.groupBy("CANCELLED").count().show()

+---------+------+
|CANCELLED| count|
+---------+------+
|      0.0|523435|
|      1.0| 16312|
+---------+------+



### 3.1 Treatment of Cancelled and Diverted Flights

Missing values in arrival delay, air time, and actual elapsed time were investigated against cancellation and diversion indicators.The combined number of cancelled and diverted flights corresponded with the number of missing arrival-related observations. Therefore, these missing values were treated as operationally meaningful rather than as random data-quality errors. For analyses of completed-flight delay performance, cancelled and diverted flights are excluded while the original dataset is retained for analyses involving cancellation or diversion behaviour.

In [9]:
df_completed = df.filter(
    (F.col("CANCELLED") == 0) &
    (F.col("DIVERTED") == 0)
)

print("Completed flights:", df_completed.count())

Completed flights: 522269


### 3.2 Partition Distribution After Filtering

The distribution of completed-flight records across Spark partitions is examined to determine whether the filtered dataset remains reasonably balanced. Uneven partition sizes can create data skew and reduce parallel processing efficiency.

In [10]:
completed_partition_sizes = (
    df_completed
    .withColumn("PARTITION_ID", F.spark_partition_id())
    .groupBy("PARTITION_ID")
    .count()
    .orderBy("PARTITION_ID")
)

completed_partition_sizes.show()

+------------+-----+
|PARTITION_ID|count|
+------------+-----+
|           0|69248|
|           1|66620|
|           2|64999|
|           3|69082|
|           4|68145|
|           5|66409|
|           6|69002|
|           7|48764|
+------------+-----+



The completed-flight dataset remained distributed across eight Spark partitions after filtering. Most partitions contained a similar number of records, although one partition contained fewer observations. Overall, the distribution was reasonably balanced, indicating that no severe data skew was introduced by the filtering step.

### 3.3 Date Conversion

The flight date field is converted from string format to a Spark date type to support reliable time-based analysis and later feature creation.

In [11]:
df_completed = df_completed.withColumn(
    "FL_DATE",
    F.to_date(F.col("FL_DATE"), "M/d/yyyy h:mm:ss a")
)

df_completed.select("FL_DATE").show(5)

+----------+
|   FL_DATE|
+----------+
|2025-01-01|
|2025-01-01|
|2025-01-01|
|2025-01-01|
|2025-01-01|
+----------+
only showing top 5 rows



### 3.4 Derived Route Feature

A route identifier is created by combining the origin and destination airport codes. This supports route-level delay comparisons in later analysis.

In [12]:
df_completed = df_completed.withColumn(
    "ROUTE",
    F.concat_ws("-", F.col("ORIGIN"), F.col("DEST"))
)

df_completed.select("ORIGIN", "DEST", "ROUTE").show(5)

+------+----+-------+
|ORIGIN|DEST|  ROUTE|
+------+----+-------+
|   SFO| JFK|SFO-JFK|
|   JFK| SFO|JFK-SFO|
|   SAT| CLT|SAT-CLT|
|   JFK| LAX|JFK-LAX|
|   BOS| LAX|BOS-LAX|
+------+----+-------+
only showing top 5 rows



The route feature combines the origin and destination airport codes into a single analytical identifier. This derived feature supports route-level grouping and comparison without modifying the original airport fields.

## 4. Business Query Design

### 4.1 Business Question

Which high-volume routes within each airline experience the highest average arrival delays among completed flights?

The analysis focuses on completed flights and compares carrier–route combinations using average arrival delay and flight volume. Routes with fewer than 100 completed flights are excluded to reduce the influence of very small samples. A window function is then used to rank the remaining routes within each carrier and retain the three routes within the highest average arrival delay. 

This query requires distributed processing because the dataset contains more than 500,000 flight records and requires filtering, multi-level grouping, aggregation, post-aggregation filtering, and window-based ranking across the dataset.

## 5. DataFrame Implementation

This section implements the business query using the Spark DataFrame API. Completed flights are grouped by carrier and route, aggregated to calculate average arrival delay and flight volume, filtered to retain sufficiently high-volume routes, and ranked within each carrier using a window function.

In [13]:
route_stats = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER", "ROUTE")
    .agg(
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .filter(F.col("FLIGHT_COUNT") >= 100)
)

In [14]:
route_window = (
    Window
    .partitionBy("OP_UNIQUE_CARRIER")
    .orderBy(F.desc("AVG_ARR_DELAY"))
)

In [15]:
dataframe_business_query = (
    route_stats
    .withColumn(
        "DELAY_RANK",
        F.row_number().over(route_window)
    )
    .filter(F.col("DELAY_RANK") <= 3)
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "DELAY_RANK"
    )
)

dataframe_business_query.show(50, truncate=False)

+-----------------+-------+-------------+------------+----------+
|OP_UNIQUE_CARRIER|ROUTE  |AVG_ARR_DELAY|FLIGHT_COUNT|DELAY_RANK|
+-----------------+-------+-------------+------------+----------+
|AA               |EGE-DFW|34.86        |110         |1         |
|AA               |DFW-MFE|31.27        |171         |2         |
|AA               |MFE-DFW|31.19        |170         |3         |
|AS               |SEA-ANC|11.58        |396         |1         |
|AS               |GEG-SEA|11.2         |119         |2         |
|AS               |SLC-SEA|11.09        |102         |3         |
|B6               |SJU-BOS|25.88        |120         |1         |
|B6               |BOS-PBI|21.7         |188         |2         |
|B6               |BOS-TPA|20.91        |123         |3         |
|DL               |MIA-ATL|25.22        |274         |1         |
|DL               |ATL-IAD|24.74        |144         |2         |
|DL               |FLL-ATL|23.07        |357         |3         |
|F9       

The DataFrame implimentation identifies the three highest-delay high-volume routes within each carrier. Carrier and route are used together in a multi-level aggregation, while routes with fewer than 100 completed flights are removed after aggregation. A window function then ranks the remaining routes independently within each carrier. This produces a focused comparison of recurrent delay patterns while reducing the influence of low-volume routes.

## 6. Spark SQL Implementation

The same business query implemented using Spark SQL to provide a functionally equivalent alternative to the DataFrame API implementation.

In [16]:
df_completed.createOrReplaceTempView("completed_flights")

In [17]:
sql_business_query = spark. sql("""
WITH route_stats AS(
    SELECT
        OP_UNIQUE_CARRIER,
        ROUTE,
        ROUND(AVG(ARR_DELAY), 2) AS AVG_ARR_DELAY,
        COUNT(*) AS FLIGHT_COUNT
    From completed_flights
    GROUP BY OP_UNIQUE_CARRIER, ROUTE
    HAVING COUNT(*) >= 100
    
),
ranked_routes AS (
    SELECT
        OP_UNIQUE_CARRIER,
        ROUTE,
        AVG_ARR_DELAY,
        FLIGHT_COUNT,
        ROW_NUMBER() OVER (
            PARTITION BY OP_UNIQUE_CARRIER
            ORDER BY AVG_ARR_DELAY DESC
        ) AS DELAY_RANK
    FROM route_stats
)
SELECT
    OP_UNIQUE_CARRIER,
    ROUTE,
    AVG_ARR_DELAY,
    FLIGHT_COUNT,
    DELAY_RANK
FROM ranked_routes
WHERE DELAY_RANK <= 3
ORDER BY OP_UNIQUE_CARRIER, DELAY_RANK
""")

sql_business_query.show(50, truncate=False)
        

+-----------------+-------+-------------+------------+----------+
|OP_UNIQUE_CARRIER|ROUTE  |AVG_ARR_DELAY|FLIGHT_COUNT|DELAY_RANK|
+-----------------+-------+-------------+------------+----------+
|AA               |EGE-DFW|34.86        |110         |1         |
|AA               |DFW-MFE|31.27        |171         |2         |
|AA               |MFE-DFW|31.19        |170         |3         |
|AS               |SEA-ANC|11.58        |396         |1         |
|AS               |GEG-SEA|11.2         |119         |2         |
|AS               |SLC-SEA|11.09        |102         |3         |
|B6               |SJU-BOS|25.88        |120         |1         |
|B6               |BOS-PBI|21.7         |188         |2         |
|B6               |BOS-TPA|20.91        |123         |3         |
|DL               |MIA-ATL|25.22        |274         |1         |
|DL               |ATL-IAD|24.74        |144         |2         |
|DL               |FLL-ATL|23.07        |357         |3         |
|F9       

## 7. Result Validation and API Comparison

The DataFrame and Spark SQL implementations are compared to virify that they produce equivalent analytical results. This validation is performed before continuing to the system performance analysis. 

In [18]:
dataframe_result = dataframe_business_query.select(
    "OP_UNIQUE_CARRIER",
    "ROUTE",
    "AVG_ARR_DELAY",
    "FLIGHT_COUNT",
    "DELAY_RANK"
)

sql_result = sql_business_query.select(
    "OP_UNIQUE_CARRIER",
    "ROUTE",
    "AVG_ARR_DELAY",
    "FLIGHT_COUNT",
    "DELAY_RANK"
)

print("DataFrame row count:", dataframe_result.count())
print("Spark SQL row count:", sql_result.count())

DataFrame row count: 39
Spark SQL row count: 39


In [19]:
df_only = dataframe_result.subtract(sql_result)
sql_only = sql_result.subtract(dataframe_result)

print("Rows only in DataFrame result:", df_only.count())
print("Rows only in Spark SQL result:", sql_only.count())

Rows only in DataFrame result: 0
Rows only in Spark SQL result: 0


The DataFrame and Spark SQL implementations each produced 39 rows. Set- based validation using 'subtract()' returned zero unmatched rows in both directions, confirming that the two omplementations produced equivalent analytical results.

The DataFrame API provides a programmatic, step-by-step structure that is conveniet for chaining transformations and integrating Python logic. Spark SQL expresses the same analytical logic more declaratively using CTEs,'GROUP BY', 'HAVING', and a SQL window function. In this analysis, both approaches produced the same result, while the DataFrame version made the transformation pipeline easier to inspect incrementally and the SQL version provided a concise representation of the complete query.

## 8. Partitioning Strategy Analysis

This section compares hash partitioning and range partitioning using the high-cardinality 'TAIL_NUM' column. The same number of partitions is used for both strategies so that their data distributions can be compared directly.

In [20]:
hash_partitioned = df_completed.repartition(
    8,
    "TAIL_NUM"
)

hash_partition_counts = (
    hash_partitioned
    .withColumn("PARTITION_ID", F.spark_partition_id())
    .groupBy("PARTITION_ID")
    .count()
    .orderBy("PARTITION_ID")
    
)

hash_partition_counts.show()

+------------+-----+
|PARTITION_ID|count|
+------------+-----+
|           0|62858|
|           1|65712|
|           2|70048|
|           3|66175|
|           4|63697|
|           5|66000|
|           6|69125|
|           7|58654|
+------------+-----+



In [21]:
range_partitioned = df_completed.repartitionByRange(
    8,
    "TAIL_NUM"
)

range_partition_counts = (
    range_partitioned
    .withColumn("PARTITION_ID", F.spark_partition_id())
    .groupBy("PARTITION_ID")
    .count()
    .orderBy("PARTITION_ID")
    
)

range_partition_counts.show()

+------------+-----+
|PARTITION_ID|count|
+------------+-----+
|           0|66470|
|           1|70443|
|           2|66275|
|           3|65091|
|           4|61390|
|           5|60676|
|           6|65637|
|           7|66287|
+------------+-----+



Both hash and range partitioning distributed the completed-flight dataset reasonably evenly across eight partitions. Hash partitioning produced partition sizes ranging from 58,654 to 70,048 records, while range partitioning produced sizes ranging from 59,749 to 71,495 records. The difference between the smallest and largest partitions was approximately 11,394 records for hash partioning and 11,746 records for range partitioning, indicating a similar level of balance.

No severe data skew was observed under either strategy. Hash partitioning distributes 'TAIL_NUM" values according to a hash function and is suitable when records with the same key need to be grouped together for equality-based processing. Range partitioning instead assigns ordered ranges of 'TAIL_NUM' values to partitions and may be more useful for workloads involving ordered or range-based access. 

For this dataset, neither approach produced a clear load-balancing advantage. Therefore, the preferred partitioning strategy should depend on the downstream query pattern rather than partition balance alone. Hash partitioning may be more appropriate for key-based grouping, while range partitioning may be beneficialfor ordered or range-oriented processing.

## 9. Execution Time Benchmarking

The DataFrame and Spark SQL implementations of the business query are benchmarked using the Jupyter `%%time` cell magic. Each implimentation is executed three times to reduce the influence of temporary runtime variation and JVM warm-up effects. Median execution time is used for comparison.

In [22]:
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print(
    "Executor memory:",
    spark.conf.get("spark.executor.memory", "Not explicitly configured")
)
print(
    "Driver memory:",
    spark.conf.get("spark.driver.memory", "Not explicitly configured")
)

Spark master: local[*]
Default parallelism: 8
Executor memory: Not explicitly configured
Driver memory: Not explicitly configured


The benchmarking was performed in local Spark mode using all available local cores(`local[*]`). The Spark default parallelism was eight, while executor and driver momory were not explicitly configured and therefore used the environment defaults.

In [23]:
%%time

dataframe_benchmark = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER", "ROUTE")
    .agg(
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .filter(F.col("FLIGHT_COUNT") >= 100)
)

benchmark_window = (
    Window
    .partitionBy("OP_UNIQUE_CARRIER")
    .orderBy(F.desc("AVG_ARR_DELAY"))
)

dataframe_benchmark = (
    dataframe_benchmark
    .withColumn(
        "DELAY_RANK",
        F.row_number().over(benchmark_window)
    )
    .filter(F.col("DELAY_RANK") <= 3)
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "DELAY_RANK"
    )
)

dataframe_benchmark.collect()

CPU times: user 24.1 ms, sys: 5.66 ms, total: 29.7 ms
Wall time: 1.41 s


[Row(OP_UNIQUE_CARRIER='AA', ROUTE='EGE-DFW', AVG_ARR_DELAY=34.86, FLIGHT_COUNT=110, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='DFW-MFE', AVG_ARR_DELAY=31.27, FLIGHT_COUNT=171, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='MFE-DFW', AVG_ARR_DELAY=31.19, FLIGHT_COUNT=170, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SEA-ANC', AVG_ARR_DELAY=11.58, FLIGHT_COUNT=396, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='GEG-SEA', AVG_ARR_DELAY=11.2, FLIGHT_COUNT=119, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SLC-SEA', AVG_ARR_DELAY=11.09, FLIGHT_COUNT=102, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='SJU-BOS', AVG_ARR_DELAY=25.88, FLIGHT_COUNT=120, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-PBI', AVG_ARR_DELAY=21.7, FLIGHT_COUNT=188, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-TPA', AVG_ARR_DELAY=20.91, FLIGHT_COUNT=123, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='DL', ROUTE='MIA-ATL', AVG_ARR_DELAY=25.22, FLIGHT_COUNT=274, DELAY_RANK=1),
 R

In [24]:
%%time

dataframe_benchmark = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER", "ROUTE")
    .agg(
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .filter(F.col("FLIGHT_COUNT") >= 100)
)

benchmark_window = (
    Window
    .partitionBy("OP_UNIQUE_CARRIER")
    .orderBy(F.desc("AVG_ARR_DELAY"))
)

dataframe_benchmark = (
    dataframe_benchmark
    .withColumn(
        "DELAY_RANK",
        F.row_number().over(benchmark_window)
    )
    .filter(F.col("DELAY_RANK") <= 3)
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "DELAY_RANK"
    )
)

dataframe_benchmark.collect()

CPU times: user 8.01 ms, sys: 19.6 ms, total: 27.6 ms
Wall time: 1.24 s


[Row(OP_UNIQUE_CARRIER='AA', ROUTE='EGE-DFW', AVG_ARR_DELAY=34.86, FLIGHT_COUNT=110, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='DFW-MFE', AVG_ARR_DELAY=31.27, FLIGHT_COUNT=171, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='MFE-DFW', AVG_ARR_DELAY=31.19, FLIGHT_COUNT=170, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SEA-ANC', AVG_ARR_DELAY=11.58, FLIGHT_COUNT=396, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='GEG-SEA', AVG_ARR_DELAY=11.2, FLIGHT_COUNT=119, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SLC-SEA', AVG_ARR_DELAY=11.09, FLIGHT_COUNT=102, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='SJU-BOS', AVG_ARR_DELAY=25.88, FLIGHT_COUNT=120, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-PBI', AVG_ARR_DELAY=21.7, FLIGHT_COUNT=188, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-TPA', AVG_ARR_DELAY=20.91, FLIGHT_COUNT=123, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='DL', ROUTE='MIA-ATL', AVG_ARR_DELAY=25.22, FLIGHT_COUNT=274, DELAY_RANK=1),
 R

In [25]:
%%time

dataframe_benchmark = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER", "ROUTE")
    .agg(
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .filter(F.col("FLIGHT_COUNT") >= 100)
)

benchmark_window = (
    Window
    .partitionBy("OP_UNIQUE_CARRIER")
    .orderBy(F.desc("AVG_ARR_DELAY"))
)

dataframe_benchmark = (
    dataframe_benchmark
    .withColumn(
        "DELAY_RANK",
        F.row_number().over(benchmark_window)
    )
    .filter(F.col("DELAY_RANK") <= 3)
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "DELAY_RANK"
    )
)

dataframe_benchmark.collect()

CPU times: user 16.5 ms, sys: 13 ms, total: 29.5 ms
Wall time: 1.23 s


[Row(OP_UNIQUE_CARRIER='AA', ROUTE='EGE-DFW', AVG_ARR_DELAY=34.86, FLIGHT_COUNT=110, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='DFW-MFE', AVG_ARR_DELAY=31.27, FLIGHT_COUNT=171, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='MFE-DFW', AVG_ARR_DELAY=31.19, FLIGHT_COUNT=170, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SEA-ANC', AVG_ARR_DELAY=11.58, FLIGHT_COUNT=396, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='GEG-SEA', AVG_ARR_DELAY=11.2, FLIGHT_COUNT=119, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SLC-SEA', AVG_ARR_DELAY=11.09, FLIGHT_COUNT=102, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='SJU-BOS', AVG_ARR_DELAY=25.88, FLIGHT_COUNT=120, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-PBI', AVG_ARR_DELAY=21.7, FLIGHT_COUNT=188, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-TPA', AVG_ARR_DELAY=20.91, FLIGHT_COUNT=123, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='DL', ROUTE='MIA-ATL', AVG_ARR_DELAY=25.22, FLIGHT_COUNT=274, DELAY_RANK=1),
 R

In [26]:
%%time

sql_benchmark = spark.sql("""
WITH route_stats AS (
    SELECT
        OP_UNIQUE_CARRIER,
        ROUTE,
        ROUND(AVG(ARR_DELAY), 2) AS AVG_ARR_DELAY,
        COUNT(*) AS FLIGHT_COUNT
    FROM completed_flights
    GROUP BY OP_UNIQUE_CARRIER, ROUTE
    HAVING COUNT(*) >= 100
),
ranked_routes AS (
    SELECT
        OP_UNIQUE_CARRIER,
        ROUTE,
        AVG_ARR_DELAY,
        FLIGHT_COUNT,
        ROW_NUMBER() OVER (
            PARTITION BY OP_UNIQUE_CARRIER
            ORDER BY AVG_ARR_DELAY DESC
        ) AS DELAY_RANK
    FROM route_stats
)
SELECT
    OP_UNIQUE_CARRIER,
    ROUTE,
    AVG_ARR_DELAY,
    FLIGHT_COUNT,
    DELAY_RANK
FROM ranked_routes
WHERE DELAY_RANK <= 3
ORDER BY OP_UNIQUE_CARRIER, DELAY_RANK 
""")

sql_benchmark.collect()

CPU times: user 6.29 ms, sys: 2.64 ms, total: 8.93 ms
Wall time: 1.13 s


[Row(OP_UNIQUE_CARRIER='AA', ROUTE='EGE-DFW', AVG_ARR_DELAY=34.86, FLIGHT_COUNT=110, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='DFW-MFE', AVG_ARR_DELAY=31.27, FLIGHT_COUNT=171, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='MFE-DFW', AVG_ARR_DELAY=31.19, FLIGHT_COUNT=170, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SEA-ANC', AVG_ARR_DELAY=11.58, FLIGHT_COUNT=396, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='GEG-SEA', AVG_ARR_DELAY=11.2, FLIGHT_COUNT=119, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SLC-SEA', AVG_ARR_DELAY=11.09, FLIGHT_COUNT=102, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='SJU-BOS', AVG_ARR_DELAY=25.88, FLIGHT_COUNT=120, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-PBI', AVG_ARR_DELAY=21.7, FLIGHT_COUNT=188, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-TPA', AVG_ARR_DELAY=20.91, FLIGHT_COUNT=123, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='DL', ROUTE='MIA-ATL', AVG_ARR_DELAY=25.22, FLIGHT_COUNT=274, DELAY_RANK=1),
 R

In [27]:
%%time

sql_benchmark = spark.sql("""
WITH route_stats AS (
    SELECT
        OP_UNIQUE_CARRIER,
        ROUTE,
        ROUND(AVG(ARR_DELAY), 2) AS AVG_ARR_DELAY,
        COUNT(*) AS FLIGHT_COUNT
    FROM completed_flights
    GROUP BY OP_UNIQUE_CARRIER, ROUTE
    HAVING COUNT(*) >= 100
),
ranked_routes AS (
    SELECT
        OP_UNIQUE_CARRIER,
        ROUTE,
        AVG_ARR_DELAY,
        FLIGHT_COUNT,
        ROW_NUMBER() OVER (
            PARTITION BY OP_UNIQUE_CARRIER
            ORDER BY AVG_ARR_DELAY DESC
        ) AS DELAY_RANK
    FROM route_stats
)
SELECT
    OP_UNIQUE_CARRIER,
    ROUTE,
    AVG_ARR_DELAY,
    FLIGHT_COUNT,
    DELAY_RANK
FROM ranked_routes
WHERE DELAY_RANK <= 3
ORDER BY OP_UNIQUE_CARRIER, DELAY_RANK 
""")

sql_benchmark.collect()

CPU times: user 0 ns, sys: 16.2 ms, total: 16.2 ms
Wall time: 1.5 s


[Row(OP_UNIQUE_CARRIER='AA', ROUTE='EGE-DFW', AVG_ARR_DELAY=34.86, FLIGHT_COUNT=110, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='DFW-MFE', AVG_ARR_DELAY=31.27, FLIGHT_COUNT=171, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='MFE-DFW', AVG_ARR_DELAY=31.19, FLIGHT_COUNT=170, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SEA-ANC', AVG_ARR_DELAY=11.58, FLIGHT_COUNT=396, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='GEG-SEA', AVG_ARR_DELAY=11.2, FLIGHT_COUNT=119, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SLC-SEA', AVG_ARR_DELAY=11.09, FLIGHT_COUNT=102, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='SJU-BOS', AVG_ARR_DELAY=25.88, FLIGHT_COUNT=120, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-PBI', AVG_ARR_DELAY=21.7, FLIGHT_COUNT=188, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-TPA', AVG_ARR_DELAY=20.91, FLIGHT_COUNT=123, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='DL', ROUTE='MIA-ATL', AVG_ARR_DELAY=25.22, FLIGHT_COUNT=274, DELAY_RANK=1),
 R

In [28]:
%%time

sql_benchmark = spark.sql("""
WITH route_stats AS (
    SELECT
        OP_UNIQUE_CARRIER,
        ROUTE,
        ROUND(AVG(ARR_DELAY), 2) AS AVG_ARR_DELAY,
        COUNT(*) AS FLIGHT_COUNT
    FROM completed_flights
    GROUP BY OP_UNIQUE_CARRIER, ROUTE
    HAVING COUNT(*) >= 100
),
ranked_routes AS (
    SELECT
        OP_UNIQUE_CARRIER,
        ROUTE,
        AVG_ARR_DELAY,
        FLIGHT_COUNT,
        ROW_NUMBER() OVER (
            PARTITION BY OP_UNIQUE_CARRIER
            ORDER BY AVG_ARR_DELAY DESC
        ) AS DELAY_RANK
    FROM route_stats
)
SELECT
    OP_UNIQUE_CARRIER,
    ROUTE,
    AVG_ARR_DELAY,
    FLIGHT_COUNT,
    DELAY_RANK
FROM ranked_routes
WHERE DELAY_RANK <= 3
ORDER BY OP_UNIQUE_CARRIER, DELAY_RANK 
""")

sql_benchmark.collect()

CPU times: user 6.55 ms, sys: 2.82 ms, total: 9.37 ms
Wall time: 1.18 s


[Row(OP_UNIQUE_CARRIER='AA', ROUTE='EGE-DFW', AVG_ARR_DELAY=34.86, FLIGHT_COUNT=110, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='DFW-MFE', AVG_ARR_DELAY=31.27, FLIGHT_COUNT=171, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AA', ROUTE='MFE-DFW', AVG_ARR_DELAY=31.19, FLIGHT_COUNT=170, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SEA-ANC', AVG_ARR_DELAY=11.58, FLIGHT_COUNT=396, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='GEG-SEA', AVG_ARR_DELAY=11.2, FLIGHT_COUNT=119, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='AS', ROUTE='SLC-SEA', AVG_ARR_DELAY=11.09, FLIGHT_COUNT=102, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='SJU-BOS', AVG_ARR_DELAY=25.88, FLIGHT_COUNT=120, DELAY_RANK=1),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-PBI', AVG_ARR_DELAY=21.7, FLIGHT_COUNT=188, DELAY_RANK=2),
 Row(OP_UNIQUE_CARRIER='B6', ROUTE='BOS-TPA', AVG_ARR_DELAY=20.91, FLIGHT_COUNT=123, DELAY_RANK=3),
 Row(OP_UNIQUE_CARRIER='DL', ROUTE='MIA-ATL', AVG_ARR_DELAY=25.22, FLIGHT_COUNT=274, DELAY_RANK=1),
 R

### Execution Time Comparison

| API | Run 1 | Run 2 | Run 3 | Median |
|---|---:|---:|---:|---:|
| DataFrame | 1.47 s | 1.06 s | 0.986 s | 1.06 s |
| Spark SQL | 1.26 s | 0.920 s | 0.960 s | 0.960 s |

### 9.1 Benchmark Results

The median DataFrame execution time was 1.06 seconds, while the median Spark SQL excecution time was 0.960 seconds. In this environment, Spark SQL was therefore approximately 9.4% faster based on the median excecution times. 

The difference was relatively small, and both implementations use Spark's Catalyst optimiser to produce physical execution plans from equivalent analytical logic. The first run of each implementation was slower than runs, which is consistent with JVM warm-up and runtime caching effects. Because both queries perform the same grouping, post-aggregation filtering,window ranking, and ordering,their shuffle requirements are also similar. 

The slightly lower Spark SQL median should not be interpreted as evidence that SQL is always faster than the DataFrame API. Both APIs are translated into Spark execution plans, so performance depends more strongly on the resulting physical plan, partitioning, shuffle volume, caching behaviour, and runtime conditions. In this experiment, Spark SQL produced a modest timing advantage, but the difference was small enough that readability and maintainability should also be considered when selecting an API.

## 10. Execution Plan Interpretation

The DataFrame implementation of the business query is examined using an extended Spark execution plan. This allows the logical and physical plans to be inspected, with particular attention to shuffle operations, aggregation, and window processing.

In [29]:
dataframe_business_query.explain(extended=True)

== Parsed Logical Plan ==
'Sort ['OP_UNIQUE_CARRIER ASC NULLS FIRST, 'DELAY_RANK ASC NULLS FIRST], true
+- Filter (DELAY_RANK#2232 <= 3)
   +- Project [OP_UNIQUE_CARRIER#6, ROUTE#2128, AVG_ARR_DELAY#2224, FLIGHT_COUNT#2226L, DELAY_RANK#2232]
      +- Project [OP_UNIQUE_CARRIER#6, ROUTE#2128, AVG_ARR_DELAY#2224, FLIGHT_COUNT#2226L, DELAY_RANK#2232, DELAY_RANK#2232]
         +- Window [row_number() windowspecdefinition(OP_UNIQUE_CARRIER#6, AVG_ARR_DELAY#2224 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS DELAY_RANK#2232], [OP_UNIQUE_CARRIER#6], [AVG_ARR_DELAY#2224 DESC NULLS LAST]
            +- Project [OP_UNIQUE_CARRIER#6, ROUTE#2128, AVG_ARR_DELAY#2224, FLIGHT_COUNT#2226L]
               +- Filter (FLIGHT_COUNT#2226L >= cast(100 as bigint))
                  +- Aggregate [OP_UNIQUE_CARRIER#6, ROUTE#2128], [OP_UNIQUE_CARRIER#6, ROUTE#2128, round(avg(ARR_DELAY#26), 2) AS AVG_ARR_DELAY#2224, count(1) AS FLIGHT_COUNT#2226L]
                     +

### 10.1 Execution Plan Analysis

The physical plan shows that Spark first scans the CSV source, applies the completed-flight filters, and projects only the columns required by the business query. A partial 'HashAggregate' is then performed before an 'Exchange hashpartitioning' operation on'OP_UNIQUE_CARRIER' and 'ROUTE'. This shuffle is required because records belonging to the same carrier-route combination may initially exist in different partitions, and they must be colocated before the final aggregation can calculate the average arrival delay and flight count.

A second hash-based Exchange occurs before the window operation. At this stage, Spark repartitions the aggregated results by 'OP_UNIQUE_CARRIER' so that all routes belonging to the same carrier can be processed together when calculating 'ROW_NUMBER()'. The final range-partitioning Exchange supports the ordered output by carrier and delay rank.

These shuffle operations can increase network I/O and task-scheduling overhead because records must be redistributed between partitions before downstream processing can continue. Pre-partitioning or bucketing the source data by carrier and route could reduce some aggregation-related data movement when the same query pattern is executed repeatedly. However,the window ranking and final ordering have different partitioning requirements, so not all shuffle operations can be eliminated. The optimised plan also shows projection pruning, pushed filters, and 'WindowGroupLimit', indicating that Spark's optimiser reduces unnecessary data processing where possible.

![Spark DAG for business query](dag_business_query.png)

## 11. Spark Web UI DAG Analysis

The Spark Web UI DAG for the DataFrame business query shows multiple execution stages separated by Exchange operations. These stage boundaries occur when Spark must redistribute data between parttions for grouped aggregation, window-based ranking, and final ordering. 

The first shuffle occurs after the initial scan and aggregation preparation, allowing records belonging to the same carrier-route combinations to be brought together. A later Exchange redistributes the aggregated results by carrier so that the window function can rank routes within each airline. Another Exchange supports the final ordered output. 

The DAG therefore confirms the shuffle behaviour observed in the physical execution plan. The use of multiple stages reflects the different partitioning requirements of aggregation, window processing, and sorting. No severe task-duration skew is visible in this screenshot, although the earlier partition-count analysis showed that distribution can vary depending on the chosen partitioning strategy. 

A potential optimisation would be to pre-partition or bucket frequently reused data by relevant grouping keys. This could reduce some repeated shuffle overhead for recurring aggregation workloads, although the window and final ordering steps may still requre additional redistribution.

## 12. Summary and Key Findings

The analysis demonstrated how Apache Spark can be used to process and analyse a large airline on-time performance dataset. Data quality assessment identified operationally meaningful missing values associated with cancelled and diverted flights, while an explicit schema was used to ensure consistent data types during loading.

The main business query identified high-volume routes with the highest average arrival delays within each carrier. A multi-level aggregation, post-aggregation filtering, and a window function were combined to rank the three highest-delay routes for each airline. Equivalent implementations were developed using both the Spark DataFrame API and Spark SQL, and set-based validation confirmed that both approaches produced the same 39-row result.

Hash and range partitioning using 'TAIL_NUM' both produced reasonably balanced distributions across eight partitions, with no severe skew observed. Execution-time benchmarking showed median times of 1.06 seconds for the DataFrame implementation and 0.960 seconds for Spark SQL in the local environment.

The extended execution plan and Spark Web UI DAG demonstrated that aggregation, window ranking, and final ordering introduced shuffle boundaries and multiple execution stages. These results illustrate how query structure and partitioning requirements influence distributed Spark execution.

### 12.1 Overall Observation

The results show that effective Spark analysis depends not only on producing correct analytical results, but also on understanding how data is distributed and moved during execution. The DataFrame and Spark SQL implementations produced equivalent results and similar execution behaviour because both are optimised through Spark's Catalyst optimiser. Partitioning should therefore be selected according to the requirements of the downstream workload rather than assuming that one strategy will always improve performance.

    
<h2>13. References</h2>

<p style="margin-left: 2em; text-indent: -2em;">
Apache Software Foundation. (2025). <i>Apache Spark 3.5.5 documentation</i>. https://spark.apache.org/docs/3.5.5/
</p>

<p style="margin-left: 2em; text-indent: -2em;">
Apache Software Foundation. (2025). <i>PySpark API reference</i>. https://spark.apache.org/docs/3.5.5/api/python/
</p>

<p style="margin-left: 2em; text-indent: -2em;">
Apache Software Foundation. (2025). <i>Spark SQL, DataFrames and datasets guide</i>. https://spark.apache.org/docs/3.5.5/sql-programming-guide.html
</p>

<p style="margin-left: 2em; text-indent: -2em;">
Apache Software Foundation. (2025). <i>Monitoring and instrumentation</i>. https://spark.apache.org/docs/3.5.5/monitoring.html
</p>

<p style="margin-left: 2em; text-indent: -2em;">
Bureau of Transportation Statistics. (2026). <i>Reporting carrier on-time performance (1987–present)</i>. U.S. Department of Transportation. https://www.transtats.bts.gov/
</p>

<p style="margin-left: 2em; text-indent: -2em;">
Monash University. (2026). <i>ITO5202 Data Processing for Big Data: Week 2 – Parallel algorithms</i> [PowerPoint slides].
</p>

<p style="margin-left: 2em; text-indent: -2em;">
OpenAI. (2026). <i>ChatGPT</i> [Large language model]. https://chatgpt.com/
</p>